In [ ]:
import folium
import geojson
import oracledb
from shapely.wkt import dumps, loads

# Definicja i test połączenia


In [ ]:
dsn = oracledb.makedsn("dbmanage.lab.ii.agh.edu.pl", 1521, sid="DBMANAGE")

username = "student"
password = "stu638dent"

connection = oracledb.connect(user=username, password=password, dsn=dsn)

In [ ]:
print(connection)

# Wybór schematu US_SPAT


In [ ]:
query = """ALTER SESSION SET CURRENT_SCHEMA = US_SPAT"""

cursor = connection.cursor()

cursor.execute(query)

In [ ]:
cursor = connection.cursor()

query = """select id, state from us_states"""

for row in cursor.execute(query):
    print(row)

In [ ]:
def OutputTypeHandler(cursor, name, defaultType, size, precision, scale):
    if defaultType == oracledb.CLOB:
        return cursor.var(oracledb.LONG_STRING, arraysize=cursor.arraysize)


connection.outputtypehandler = OutputTypeHandler

# Przykłady


# Florida & Texas


In [ ]:
# map_view = folium.Map()

map_view = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap")

query = """SELECT  sdo_util.to_wktgeometry(geom)
       FROM us_states
       WHERE state='Florida' or state = 'Texas'"""

# query = """SELECT  sdo_util.to_wktgeometry(geom)
#        FROM us_states """

rows = loads(cursor.execute(query).fetchall())

style = {"fillColor": "blue", "color": "red"}

features = []

for row in rows:
    feature = geojson.Feature(geometry=row[0], properties={})
    features.append(feature)


feature_collection = geojson.FeatureCollection(features)

folium.GeoJson(feature_collection, style_function=lambda x: style).add_to(map_view)

map_view

# Dodatkowa warstwa z drogami


In [ ]:
query = """SELECT  sdo_util.to_wktgeometry(geom)
       FROM us_interstates"""

rows = loads(cursor.execute(query).fetchall())

interstate_style = {"color": "blue"}

features = []

for row in rows:
    feature = geojson.Feature(geometry=row[0], properties={})
    features.append(feature)


feature_collection = geojson.FeatureCollection(features)

folium.GeoJson(feature_collection, style_function=lambda x: interstate_style).add_to(
    map_view
)

map_view

# Parki wewnątrz stanu Texas


In [ ]:
map_view = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap")

query = """SELECT  sdo_util.to_wktgeometry(p.geom)
FROM us_parks p, us_states s
WHERE s.state = 'Texas'
AND SDO_INSIDE (p.geom, s.geom ) = 'TRUE'"""

rows = loads(cursor.execute(query).fetchall())

style = {"fillColor": "blue", "color": "red"}

features = []
for row in rows:
    feature = geojson.Feature(geometry=row[0], properties={})
    features.append(feature)


feature_collection = geojson.FeatureCollection(features)
folium.GeoJson(feature_collection, style_function=lambda x: style).add_to(map_view)

# map_view.show_in_browser()
map_view